# 🧠 Notebook 09: Axion Policy Engine

## 1. Purpose + Scope

This notebook demonstrates the Axion Policy Engine, the governance layer of T81:

*   **Policy Grammar Walkthrough**: Understanding S-expression policies.
*   **Resource Limits**: Defining `max_memory` and `max_steps`.
*   **Policy Denial Demonstration**: What happens when code violates policy.
*   **Trace Analysis**: Inspecting the reasons for denial.

## 2. Spec References

*   `spec/axion-kernel.md`
*   `spec/cognitive-tiers.md`
*   `tools/axion_policy_validator.py`

## 3. Determinism Tier

**Tier A (Strict Determinism)**: Policy evaluation is deterministic. A policy `P` applied to trace `T` always yields the same `ALLOW` or `DENY` verdict.

## 4. Reproducibility Setup

We use the `axion_policy_validator.py` tool located in `tools/`.

In [ ]:
import sys
import os
import subprocess

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
tool_path = os.path.join(repo_root, "tools", "axion_policy_validator.py")

if not os.path.exists(tool_path):
    print(f"❌ Tool not found: {tool_path}")
    sys.exit(1)

print(f"✅ Tool found: {tool_path}")

## 5. Policy Grammar

Policies are S-expressions. Example:

```lisp
(policy "LimitMemory"
  (rule "memory_usage" (< (memory_used) 1024))
)
```

Let's write a sample policy to a file.

In [ ]:
policy_content = """
(policy "BasicRestrictions"
  (rule "no_network" (not (call "network_send")))
  (rule "limit_steps" (< (steps) 1000))
)
"""

with open("sample_policy.axion", "w") as f:
    f.write(policy_content)

print("Created sample_policy.axion")

## 6. Trace Generation (Simulation)

The validator checks a JSON trace against the policy. Let's create a passing trace and a failing trace.

In [ ]:
import json

passing_trace = {
    "events": [
        {"type": "step", "count": 500},
        {"type": "call", "target": "math_add"}
    ]
}

failing_trace = {
    "events": [
        {"type": "step", "count": 1500},  # > 1000 limit
        {"type": "call", "target": "network_send"} # Forbidden
    ]
}

with open("passing_trace.json", "w") as f:
    json.dump(passing_trace, f)

with open("failing_trace.json", "w") as f:
    json.dump(failing_trace, f)

print("Created trace files.")

## 7. Policy Validation

We run the validator tool against our traces.

In [ ]:
def run_validation(policy_file, trace_file):
    cmd = [sys.executable, tool_path, "--policy", policy_file, "--trace", trace_file]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return result.stdout, result.stderr, result.returncode

print("--- Validating Passing Trace ---")
out, err, code = run_validation("sample_policy.axion", "passing_trace.json")
if code == 0:
    print("✅ Validation PASSED")
else:
    print("❌ Validation FAILED")
    print(err)

print("\n--- Validating Failing Trace ---")
out, err, code = run_validation("sample_policy.axion", "failing_trace.json")
if code != 0:
    print("✅ Validation correctly FAILED (as expected)")
    print("Output:", out)
else:
    print("❌ Validation unexpectedly PASSED")

## 8. Failure Mode Demonstration

The output above shows how policy violations are reported.

In [ ]:
# Cleanup
if os.path.exists("sample_policy.axion"):
    os.remove("sample_policy.axion")
if os.path.exists("passing_trace.json"):
    os.remove("passing_trace.json")
if os.path.exists("failing_trace.json"):
    os.remove("failing_trace.json")

## 9. Architectural Commentary

Axion decouples *execution capability* from *execution permission*. The VM can technically execute a network call, but the Axion policy wrapper denies it before it happens, or invalidates the block if the trace shows it happened.